In [3]:
# Demo: escalar, vector, matriz en Python puro
a = 3.0
x = [1.0, -2.0, 0.5]
X = [
    [1.0,  0.0, 0.5],
    [0.0,  1.0, 1.5],
    [1.0, -1.0, 2.0],
] #Esta nomenclatura suele ser la típica.

print("a:", a, ";", type(a))
print("x:", x, type(x), "; longitud:", len(x))
print("Nº filas X:", len(X), "; longitud fila 0:", len(X[0]))

print("\nIndexado:")
print("x[0] =", x[0])
print("X[2][1] =", X[2][1])

print("\nSlicing:")
print("x[1:] =", x[1:]) #Para acceder a uno o N valores del tensor. El 1 es inclusive (RECORDAR QUE SE EMPIEZA POR 0)
print("Fila 0 de X:", X[0]) #Una fila completa de una matriz se hace con indexado.

a: 3.0 ; <class 'float'>
x: [1.0, -2.0, 0.5] <class 'list'> ; longitud: 3
Nº filas X: 3 ; longitud fila 0: 3

Indexado:
x[0] = 1.0
X[2][1] = -1.0

Slicing:
x[1:] = [-2.0, 0.5]
Fila 0 de X: [1.0, 0.0, 0.5]


Como DRL prácticamente todo son tensores, este primer fragmento de código es una pequeña introducción para entender qué son. Primero se definen 3 varibles: un escalar, una vector (una lista), y una matriz (una lista de listas).

Lo de x[1:] se llama Slicing, y es para acceder desde el 1 hasta el último elemento. De momento el código es muy simple.

In [4]:
def dot_pure(w, x):
    assert len(w) == len(x), "w y x deben tener la misma longitud" #Un asset es un "test" que hace para que en caso de que no se cumpla lance un error.
    #En este caso, es para asegurarnos de que el vector de entrada y el de los pesos tienen la misma longitud.
    s = 0.0
    for wi, xi in zip(w, x): #Con zip() sacas los elementos a pares
        s += wi * xi
    return s

def linear_combination_pure(x, w, b): #Esta función ejecuta la lógica de la anterior y suma el bias.
    return dot_pure(w, x) + b

La función dot_pure define el producto escalar entre dos vectores. Primero nos aseguramos de que los dos vectores con los que se van a hacer el producto tienen la misma longitud. Esto se hace con el assert.

Luego, se inicia un contandor en el que se va a acumlar la suma, porque el resultado de un producto escalar es un número. Luego, por cada par de elementos que hay en los vectores, se hacen las multiplicaciones y se suman. Así, conseguimos el resultado final del producto. Finalmente, se devuelve.

Como la multiplicación se hace por pares, se utiliza la función zip(vector1, vector2), que hace justamente eso.


Luego se define otra función, la de linear_combination_pure, que lo unico que hace es sumar el bias al resultado del producto escalar.

¿Porqué definimos estas dos funciones?

Porque la lógica que sigue una red neuronal es la siguiente:

- Entra la entrada (valga la redundancia). La entrada no es más que un vector, y puede representar muchas cosas, desde una imagen hasta un texto embedido.
- La entrada se por los pesos con el producto escalar
- Se le suma el bias y se consigue la salida. IMPORTANTE: LA SALIDA ES LA "Z". Esta "z" aparecerá muchas veces por ahí.

Una neurona es literalmente una función tipo y = w*x + b. Eso es todo, en el PPT de clase se explica porqué se elige esta función tan simple. 

Luego hay otro concepto que veremos ahora más adelante, la función de activación, que decide se una neurona se enciende o no. Por ejemplo, cuando veo por mis ojos, no me hace falta activar las neuronas que controlan lo que huele mi nariz. Pues aquí sería lo mismo. Esta función de activación se aplicaría sobre la "z", la salida del producto escalar.

In [5]:
w = [0.8, -0.2, 0.1]
x = [1.0, 0.0, 0.5]
b = -0.1

z = linear_combination_pure(x, w, b)
print("w =", w)
print("x =", x)
print("b =", b)
print("\nz = w·x + b =", z)

w = [0.8, -0.2, 0.1]
x = [1.0, 0.0, 0.5]
b = -0.1

z = w·x + b = 0.7500000000000001


In [7]:
def step_pure(z, threshold=0.0):
    return 1 if z > threshold else 0 #Esta sería una función de activación. La neurona se activa si llega a un "threshold". Es decir, es la Umbral.

En esta celda se implementa la función de activación más simple posible: la de umbral. Es decir, la neurona se activará si llega a cierto umbral/valor fijado.

La función step_pure implementa esta función. Como entradas tiene la z, el resultado del producto escalar, y el threshold, es decir, el umbral. En este caso, la función es:
- Si el valor de z es mayor al threshold, devuelve 1.

¿Porqué implementar funciones de activación?

- Con esta neurona (el perceptrón clásico), tienes un clasificador binario básico. 

- Las funciones de activación se introducen para introducir no linealidades a la red. Sin ellas, el modelo sería un modeo lineal simple, porque sería la suma de muchas funciones lineales. Por eso, con estas funciones de activación, puedes hacer cosas infinitamente más complejas.

Un problema que trae esta función de activación es que no es derivable. Como las redes neuronales se entrenan con gradient descent, tienes que calcular la derivada de la función de pérdida L. Esta función de pérdida no es más que una función que representa el error que comete el modelo. En gradient descent, la idea es minimizar esa función de pérdida buscando el mínimo, y buscar el mínimo no es más que derivar e igualar a 0.

Aprender es minimizar la función de pérdida, y eso se hace cambiando los pesos. Por lo tanto, la derivada que hay que hacer el dL/dW. Por ahí se puede meter el LR y toda la pesca, pero de momento vamos a lo simple.


Como esta función de activación no es derivable, y además es un valor cte, en casi todos los puntos la derivada será 0, haciendo que la red no sepa si está aprendiendo, porque el descenso del gradiente es nulo. Por eso, es recomendable utilizar otras funciones de activación.



Esto de las funciones de activación y el descenso del gradiente se puede relacionar con el concepto de "backpropragation". El back propragation calcula todas las dL/dw de forma eficiente. Es decir, en una red neuronal del tipo w1*x + w2*x + w3*x +b, el descenso del gradiente se hace probando a cambiar los pesos w1, w2 y w3. Para que no sea extremadamente pesado lo de calcular todas las derivadas por separado, se hace el back propagation. Los pasos a seguir son los siguientes:

- Haces que la entrada pase por toda la red neuronal y consigues la salida yhat (la predicción). Esto es el "forward", es decir, calcular la predicción de la red,

- Calculas la función de pérdida, por ejemplo L= (y- yhat) (real vs predicho).

- Propagas el error hacia atrás en la red (el back), es decir, calcular las derivadas de dL/dw y ves qué neuronas contribuyen al error.

- Con el gradient desent, cambias los pesos de las neuronas para que el error sea menor.

In [8]:
y_hat = step_pure(z, threshold=0.5)
print("z =", z, "=> y_hat =", y_hat)

z = 0.7500000000000001 => y_hat = 1


In [9]:
y_hat = step_pure(0.4, threshold=0.5)
print("z =", z, "=> y_hat =", y_hat)

z = 0.7500000000000001 => y_hat = 0


In [15]:
import math
def linear_pure(z):
    return z

def saturated_linear_pure(z, min_val=0.0, max_val=1.0):
    return min(max(z, min_val), max_val)

def sigmoid_pure(z):
    return (1/(1+math.exp(-z)))


En esta celda tenemos tres funciones de activación nuevas.

- linear_pure(z): Entra z y sale z. Osea, una función identidad pura. (continua, derivable y derivada constante)

- saturated_linear_pure(z, min, max): Si el valor de entrada es menor al mínimo, devuelve el mínimo. Si es mayor que el mínimo y menor que el máximo, devuelve el valor de entrada. Si es mayor al valor máximo, devuelve el valor máximo. Lineal en el centro y plana en los extremos. En los extremos, la derivada es 0 y no se aprende nada. En el centro, es 1.

- Sigmoid_pure(z): Una función que va de 0 a 1 entre -inf a inf. En 0, vale 0.5. La sigmoide es un poco con la step, pero continua, derivable y suave. Esto permite utilziar el backpropagation. Ya no se utiliza tanto, porque z muy grande devuelve 1 y muy pequeño devuelve 0. Esto es el gradient vanishing. Es decir, puedes tener cambios grandes en "z", pero pequeños en la salida de la sigmoide. Esto hace que, al hacer backpropagation, los pesos no cambien mucho porque está saturada. Cuantas más capas ocultas, peor.

In [16]:
# Pruebas
test_z = [-3.0, -1.0, 0.0, 1.0, 3.0]
print("step:", [step_pure(v) for v in test_z])
print("linear:", [linear_pure(v) for v in test_z])
print("sat_linear:", [saturated_linear_pure(v, 0.0, 1.0) for v in test_z])
print("sigmoid:", [sigmoid_pure(v) for v in test_z])

step: [0, 0, 0, 1, 1]
linear: [-3.0, -1.0, 0.0, 1.0, 3.0]
sat_linear: [0.0, 0.0, 0.0, 1.0, 1.0]
sigmoid: [0.04742587317756678, 0.2689414213699951, 0.5, 0.7310585786300049, 0.9525741268224334]


In [17]:
# Tests de autocorrección
assert saturated_linear_pure(-10, 0, 1) == 0
assert saturated_linear_pure(0.2, 0, 1) == 0.2
assert saturated_linear_pure(10, 0, 1) == 1

s0 = sigmoid_pure(0.0)
assert abs(s0 - 0.5) < 1e-8
print("✅ Tests superados")

✅ Tests superados


In [19]:
def neuron_forward_pure(x, w, b, activation):
    #Implementación de forward de neurona devolviendo (y_hat, z)
    z = linear_combination_pure(x, w, b)
    y_hat = activation(z)
    return(y_hat, z)


En esta celda, se implimenta todo lo que hemos ido definiendo antes. Es decir, esta función ya es un paso completo de una neurona. Implementamos el calculo de lo que hace una neurona en una red neuronal. A esta función le entra la entrada (la x), los pesos de la neurona (la w), el bias (la b) y el tipo de función de activación (activation). Primero, se consigue la z con la función antes definida linear_combination_pure. Al resultado (z), se le aplica la función de activación elegida (activation) y se consigue la predicción (la y_hat). Finalmente, se devuelven y_hat y z. La z también se devuelve porque, en el backpropagation, necesitamos calcular derivadas que implican z.  

Una red neuronal, básicamente consiste en implementar este cálculo una y otra vez, tantas veces como neuronas tenga la red.

In [20]:
# Pruebas
x = [1.0, 0.0, 0.5]
w = [0.8, -0.2, 0.1]
b = -0.1

y_hat_step, z_step = neuron_forward_pure(x, w, b, activation=step_pure)
print("step -> z:", z_step, "y_hat:", y_hat_step)

y_hat_lin, z_lin = neuron_forward_pure(x, w, b, activation=linear_pure)
print("linear -> z:", z_lin, "y_hat:", y_hat_lin)

y_hat_sig, z_sig = neuron_forward_pure(x, w, b, activation=sigmoid_pure)
print("sigmoid -> z:", z_sig, "y_hat:", y_hat_sig)

step -> z: 0.7500000000000001 y_hat: 1
linear -> z: 0.7500000000000001 y_hat: 0.7500000000000001
sigmoid -> z: 0.7500000000000001 y_hat: 0.6791786991753931


In [21]:
# Tests de autocorrección
z_expected = linear_combination_pure(x, w, b)

y_hat, z = neuron_forward_pure(x, w, b, activation=linear_pure)
assert abs(z - z_expected) < 1e-12
assert abs(y_hat - z_expected) < 1e-12

y_hat_step, z_step = neuron_forward_pure(x, w, b, activation=step_pure)
assert abs(z_step - z_expected) < 1e-12
assert y_hat_step in (0, 1)
assert y_hat_step == step_pure(z_expected)

y_hat_sig, z_sig = neuron_forward_pure(x, w, b, activation=sigmoid_pure)
assert abs(z_sig - z_expected) < 1e-12
assert 0.0 < y_hat_sig < 1.0
assert abs(y_hat_sig - sigmoid_pure(z_expected)) < 1e-12

print("✅ Tests superados")

✅ Tests superados


In [22]:
def perceptron_predict_pure(x, w, b, threshold=0.0):
    z = linear_combination_pure(x, w, b)
    y_hat = step_pure(z, threshold=threshold)
    return y_hat

Esta celda de arriba implementa la función de predicción del perceptrón clásico. Esta función recibe la entrada, aplica la función de activación y decide a qué clase pertenece. Es una chorrada con un nombre rimbombante.

In [23]:
X_logic = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]
y_or = [0, 1, 1, 1]

w_or = [1.0, 1.0]
b_or = -0.5

preds = [perceptron_predict_pure(xi, w_or, b_or) for xi in X_logic]
print("preds OR:", preds, "target:", y_or)

assert preds == y_or

preds OR: [0, 1, 1, 1] target: [0, 1, 1, 1]


In [26]:
import numpy as np

a_np = np.array(3.0)
x_np = np.array([1.0, -2.0, 0.5])
X_np = np.array([
    [1.0,  0.0, 0.5],
    [0.0,  1.0, 1.5],
    [1.0, -1.0, 2.0],
])

print("a_np:", a_np, "shape:", a_np.shape, "ndim:", a_np.ndim, "dtype:", a_np.dtype)
print("\nx_np:", x_np, "shape:", x_np.shape, "ndim:", x_np.ndim, "dtype:", x_np.dtype)
print("\nX_np:", X_np, "X_np shape:", X_np.shape, "ndim:", X_np.ndim, "dtype:", X_np.dtype)

a_np: 3.0 shape: () ndim: 0 dtype: float64

x_np: [ 1.  -2.   0.5] shape: (3,) ndim: 1 dtype: float64

X_np: [[ 1.   0.   0.5]
 [ 0.   1.   1.5]
 [ 1.  -1.   2. ]] X_np shape: (3, 3) ndim: 2 dtype: float64


En este celda estoy haciendo algo parecido a lo que he hecho al principio definiendo los vectores y tal pero con numpy.

In [27]:
X = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])  # (N=4, D=2)

w = np.array([1.0, 1.0])   # (D=2,)
b = -0.5

z = X @ w + b # @ en python sirve para las multiplicaciones matriciales. Es algo "nuevo".
print("X shape:", X.shape)
print("w shape:", w.shape)
print("z:", z, "| shape:", z.shape)

X shape: (4, 2)
w shape: (2,)
z: [-0.5  0.5  0.5  1.5] | shape: (4,)


En python, si haces X @ w, haces una multiplicación matricial. Esto no es como lo anterior del dot_pure. Si X e w fuesen dos vectores, sí que sería el dot, porque sería un producto escalar. Pero en este caso, es multiplicación matricial. Es una forma de simplificar las cosas, nada más.

In [28]:
def linear_combination_np(x, w, b):
    """
    x: np.ndarray shape (D,)
    w: np.ndarray shape (D,)
    b: float
    """
    z = x @ w + b
    return z

Aquí estamos aplicando la función de linear_combination pero con numpy en vez de con python puro. Como antes, recibimos x, w y b. Hacemos el producto escalar de x y w con el @, y sumamos el sesgo "b". Finalmente, devolvemos el resultado de esa operación.

In [29]:
# Pruebas
w = [0.8, -0.2, 0.1]
x = [1.0, 0.0, 0.5]
b = -0.1

z_pure = linear_combination_pure(x, w, b)
z_np = linear_combination_np(np.array(x), np.array(w), b)
print("z_pure:", z_pure)
print("z_np  :", z_np)

z_pure: 0.7500000000000001
z_np  : 0.7500000000000001


In [30]:
# Tests de autocorrección

# 1) Coincide con la versión Python puro
assert np.allclose(
    linear_combination_np(np.array(x), np.array(w), b),
    linear_combination_pure(x, w, b)
)

# 2) Propiedades simples: si b=0 y w=[1,1,1], z es suma de x
x2 = np.array([2.0, -1.0, 0.5])
w2 = np.ones_like(x2)
z2 = linear_combination_np(x2, w2, 0.0)
assert np.allclose(z2, np.sum(x2))

# 3) Caso borde: x y w de ceros => z = b
x3 = np.zeros(4)
w3 = np.zeros(4)
b3 = 1.23
z3 = linear_combination_np(x3, w3, b3)
assert np.allclose(z3, b3)

print("✅ Tests superados")

✅ Tests superados


In [31]:
z = np.array([1.0, 2.0, 3.0])
b = 0.5
print("z + b =", z + b)

# Caso incorrecto
# b_bad = np.array([0.1, 0.2])
# print(z + b_bad)

z + b = [1.5 2.5 3.5]


Aquí simplmenente estamos viendo que, con numpy, si haces una suma entre un vector y un escalar, el escalar se suma a todos los valores.

In [32]:
def step_np(z, threshold=0.0):
    # Devolver 1 donde z > threshold y 0 en caso contrario
    return np.where(z >threshold, 1, 0) #Devuelve 1 o 0 en función de la condición.

def saturated_linear_np(z, min_val=0.0, max_val=1.0):
    # TODO: Usando np.clip, se puede implementar fácilmente.
    return np.clip(z, min_val, max_val)

def sigmoid_np(z):
    return (1/(1+np.exp(-z)))

En esta celda, definimos nuevamente las funciones de activación de step, saturated_linear y sigmoid, pero esta vez utilizando numpy. Con el where, puedes hacer que devuelva 1 o 0 en función de una condición.

Con el .clip, haces justamente eso, clipear los valores en un rango.

La sigmoide se hace exactametne igual que antes.

In [33]:
# Pruebas
z_test = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])

print("z_test:", z_test)
print("step (t=0):", step_np(z_test))
print("step (t=1):", step_np(z_test, threshold=1.0))
print("sat [0,1]:", saturated_linear_np(z_test, 0.0, 1.0))
print("sat [-1,1]:", saturated_linear_np(z_test, -1.0, 1.0))
print("sigmoid:", sigmoid_np(z_test))

# Con matriz
Z_mat = np.array([[-1.0, 0.0, 1.0],
                  [ 2.0,-2.0, 0.5]])
print("\nZ_mat:\n", Z_mat)
print("step(Z_mat):\n", step_np(Z_mat))
print("sat(Z_mat,[0,1]):\n", saturated_linear_np(Z_mat, 0.0, 1.0))
print("sigmoid(Z_mat):\n", sigmoid_np(Z_mat))

z_test: [-3. -1.  0.  1.  3.]
step (t=0): [0 0 0 1 1]
step (t=1): [0 0 0 0 1]
sat [0,1]: [0. 0. 0. 1. 1.]
sat [-1,1]: [-1. -1.  0.  1.  1.]
sigmoid: [0.04742587 0.26894142 0.5        0.73105858 0.95257413]

Z_mat:
 [[-1.   0.   1. ]
 [ 2.  -2.   0.5]]
step(Z_mat):
 [[0 0 1]
 [1 0 1]]
sat(Z_mat,[0,1]):
 [[0.  0.  1. ]
 [1.  0.  0.5]]
sigmoid(Z_mat):
 [[0.26894142 0.5        0.73105858]
 [0.88079708 0.11920292 0.62245933]]


In [34]:
# Tests de autocorrección

# 1) step: valores esperados con threshold=0
expected_step = np.array([0, 0, 0, 1, 1])
out_step = step_np(z_test)
assert out_step.shape == z_test.shape
assert np.array_equal(out_step, expected_step)

# 2) saturated_linear: recorte correcto
out_sat01 = saturated_linear_np(z_test, 0.0, 1.0)
assert out_sat01.shape == z_test.shape
assert np.allclose(out_sat01, np.array([0.0, 0.0, 0.0, 1.0, 1.0])) #Esto se utilza para evitar usar el ==, pues con el tema de la coma flotante puede dar errores.

out_satm11 = saturated_linear_np(z_test, -1.0, 1.0)
assert np.allclose(out_satm11, np.array([-1.0, -1.0, 0.0, 1.0, 1.0]))

# 3) sigmoid: propiedades básicas
out_sig = sigmoid_np(z_test)
assert out_sig.shape == z_test.shape
assert np.all((out_sig > 0.0) & (out_sig < 1.0))  # en (0,1) para valores finitos
assert abs(sigmoid_np(0.0) - 0.5) < 1e-12
assert sigmoid_np(-3.0) < sigmoid_np(-1.0) < sigmoid_np(0.0) < sigmoid_np(1.0) < sigmoid_np(3.0)

# 4) Funciona con matrices (vectorización)
Z_mat = np.array([[-1.0, 0.0, 1.0],
                  [ 2.0,-2.0, 0.5]])
assert step_np(Z_mat).shape == Z_mat.shape
assert saturated_linear_np(Z_mat, 0.0, 1.0).shape == Z_mat.shape
assert sigmoid_np(Z_mat).shape == Z_mat.shape

print("✅ Tests superados")

✅ Tests superados


In [35]:
def neuron_forward_np(X, w, b, activation):
    """
    X: np.ndarray shape (N, D)
    w: np.ndarray shape (D,)
    b: float
    activation: función vectorizada
    """
    b_np = np.array(b)
    assert X.shape[1] == w.shape[0]

    z = linear_combination_np(X, w, b)
    y_hat = activation(z)
    return y_hat, z

En esta celda se define el forward de una neurona pero con numpy. Primero se pasa el sesgo b a tipo array de numpy. Luego, se hace la combinación linear, se consigue z, se aplica la función de activación, y se consigue la predicción y_hat. Finalmente, se devuelve y_hat y "z".

In [36]:
# Pruebas
X = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])  # (N=4, D=2)
w = np.array([1.0, 1.0])   # (D=2,)
b = -0.5

y_hat_step, z = neuron_forward_np(X, w, b, activation=step_np)
print("z:", z)
print("y_hat (step):", y_hat_step)

y_hat_sig, z2 = neuron_forward_np(X, w, b, activation=sigmoid_np)
print("y_hat (sigmoid):", y_hat_sig)

# otro batch aleatorio (solo para ver shapes)
X2 = np.random.randn(5, 3)   # (N=5, D=3)
w2 = np.random.randn(3)      # (D=3,)
b2 = 0.1
y2, z2 = neuron_forward_np(X2, w2, b2, activation=sigmoid_np)
print("X2 shape:", X2.shape, "| z2 shape:", z2.shape, "| y2 shape:", y2.shape)

z: [-0.5  0.5  0.5  1.5]
y_hat (step): [0 1 1 1]
y_hat (sigmoid): [0.37754067 0.62245933 0.62245933 0.81757448]
X2 shape: (5, 3) | z2 shape: (5,) | y2 shape: (5,)


In [37]:
# Tests de autocorrección

# 1) Shapes correctas
y_hat, z = neuron_forward_np(X, w, b, activation=step_np)
assert z.shape == (X.shape[0],)
assert y_hat.shape == (X.shape[0],)

# 2) z coincide con el cálculo directo
z_expected = X @ w + b
assert np.allclose(z, z_expected)

# 3) y_hat coincide con aplicar activation a z
assert np.array_equal(y_hat, step_np(z_expected))

# 4) Funciona con otra activación (sigmoid) y respeta propiedades
y_sig, z_sig = neuron_forward_np(X, w, b, activation=sigmoid_np)
assert np.allclose(z_sig, z_expected)
assert np.all((y_sig > 0.0) & (y_sig < 1.0))

# 5) Robustez: otro batch y dimensiones
Xr = np.random.randn(7, 4)
wr = np.random.randn(4)
br = -0.3
yr, zr = neuron_forward_np(Xr, wr, br, activation=sigmoid_np)
assert zr.shape == (7,)
assert yr.shape == (7,)
assert np.allclose(zr, Xr @ wr + br)

print("✅ Tests superados")

✅ Tests superados


In [38]:

def perceptron_predict_np(X, w, b, threshold=0.0):
    # TODO: reutiliza neuron_forward_np con activation step_np (puedes usar lambda)
    y_hat, z = neuron_forward_np(X, w, b, activation=lambda z: step_np(z, threshold=threshold))
    return y_hat

Aquí lo mismo que antes con el perceptron. Se hace el forward para cada z.

In [39]:
# Pruebas
X_logic = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])

# OR
y_or = np.array([0, 1, 1, 1])
w_or = np.array([1.0, 1.0])
b_or = -0.5

# AND
y_and = np.array([0, 0, 0, 1])
w_and = np.array([1.0, 1.0])
b_and = -1.5

preds_or = perceptron_predict_np(X_logic, w_or, b_or)
preds_and = perceptron_predict_np(X_logic, w_and, b_and)
print("OR preds :", preds_or, "target:", y_or)
print("AND preds:", preds_and, "target:", y_and)

# Probar que el threshold afecta
preds_or_t1 = perceptron_predict_np(X_logic, w_or, b_or, threshold=1.0)
print("OR preds (threshold=1.0):", preds_or_t1)

OR preds : [0 1 1 1] target: [0 1 1 1]
AND preds: [0 0 0 1] target: [0 0 0 1]
OR preds (threshold=1.0): [0 0 0 1]


In [41]:
# Tests de autocorrección

# 1) Shapes y tipos básicos
preds = perceptron_predict_np(X_logic, w_or, b_or)
assert preds.shape == (X_logic.shape[0],)
assert np.issubdtype(preds.dtype, np.integer) or np.array_equal(preds, preds.astype(int))

# 2) OR y AND correctos con los pesos propuestos
assert np.array_equal(perceptron_predict_np(X_logic, w_or, b_or), y_or)
assert np.array_equal(perceptron_predict_np(X_logic, w_and, b_and), y_and)

# 3) Coherencia con neuron_forward_np + step (reutilización correcta)
y_hat, z = neuron_forward_np(X_logic, w_or, b_or, activation=lambda zz: step_np(zz, threshold=0.0))
assert np.array_equal(perceptron_predict_np(X_logic, w_or, b_or, threshold=0.0), y_hat)

# 4) El threshold cambia el resultado de forma coherente (al menos en algún punto)
preds_t0 = perceptron_predict_np(X_logic, w_or, b_or, threshold=0.0)
preds_t1 = perceptron_predict_np(X_logic, w_or, b_or, threshold=1.0)
assert np.any(preds_t0 != preds_t1)

print("✅ Tests superados")

✅ Tests superados


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
